# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** In Silico Computational Biology & Protein Topology

---
*A note before the geometry: what follows applies a simple vector idea (pairwise distance) to a real macromolecular structure (1UBQ, Ubiquitin). I turn the static atomic coordinates into a matrix of every alpha-carbon distance against every other, and the off-diagonal minima that fall out of that matrix mark residues far apart in the primary sequence yet close in 3D space: a **structural fingerprint** of the fold's tertiary-contact topology. This fingerprint is not a map of catalytic or functional sites (it can't be, since function lives in side chains, charges, and sequence context that a backbone distance matrix never sees), and it is a genuine map of geometric proximity, which is exactly the layer a fold's topology lives on before any chemistry gets added.*

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Single source of truth: the PDB parser and the distance-matrix builder live in
# src/distances.py (and are unit-tested there). The notebook imports them so the
# heatmap and the tests are produced by the exact same code.
sys.path.insert(0, str(Path.cwd().parent / "src"))
from distances import compute_distance_matrix, parse_pdb_coordinates

plt.style.use('dark_background')
sns.set_palette("mako")

### 1. One Atom Per Residue: the Alpha-Carbon Trace
I restrict the *in silico* matrix analysis to the **alpha-carbon (CA)** atoms, the standard one-atom-per-residue trace of the protein backbone (1UBQ: Ubiquitin). Side-chain atoms are omitted by design: the CA chain is already a faithful, low-dimensional summary of the fold's geometry, and adding side chains back in would answer a chemistry question, not a topology one.

In [ ]:
df_ca = parse_pdb_coordinates('../data/1ubq.pdb')
print(f"[*] Extracted {len(df_ca)} structural nodes (Alpha-Carbon Atoms).")
df_ca.head()

### 2. Spatial Transformation: Euclidean Distance Matrix
Computing every CA-against-every-CA distance gives me an $N \times N$ symmetric matrix, a 2D representation of spatial proximity along the chain, sequence position on both axes. Its off-diagonal dark bands are the interesting part: they flag **long-range tertiary contacts**, residues distant in sequence but packed close in 3D, which is exactly what the main diagonal alone can never show.

In [ ]:
coords = df_ca[['X', 'Y', 'Z']].values

# Vectorized euclidean distance matrix (symmetric, zero diagonal by construction)
dist_mat = compute_distance_matrix(coords)

# Create Heatmap. Plain 'viridis': distance 0 (diagonal & close contacts) maps to
# dark, large distances to bright yellow -> darker = spatially closer.
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    dist_mat,
    cmap="viridis",  # darker = closer (near 0 A); brighter = farther
    square=True,
    cbar_kws={'label': 'CA-CA Distance (Angstroms)'},
    ax=ax
)

ax.set_title("CA-CA Distance Matrix (1UBQ): Backbone Tertiary Contacts", fontsize=14, pad=15)
ax.set_xlabel("Amino Acid Residue Index")
ax.set_ylabel("Amino Acid Residue Index")

plt.show()

### Topological Conclusion
The globular fold of ubiquitin shows up in the heatmap as dark bands off the main diagonal, pairs of residues far apart in the primary sequence that sit at near-$0Å$ separation in space. I read these as **long-range tertiary contacts**: the signature of the chain folding back on itself, consistent with ubiquitin's known β-grasp fold (a five-stranded β-sheet packed against an α-helix), where sequence-distant β-strands pair spatially. The diagonal never lies about this; a backbone can't fake proximity it doesn't have.

What this map can't do is chemistry, and it's worth being precise about why. A CA-CA distance map establishes geometric proximity (two backbone atoms sitting close in space), but assigning specific hydrogen bonds needs donor/acceptor heavy-atom geometry at 3.5 Å or closer, information that lives on side chains this trace never included; and delineating the hydrophobic core needs side-chain and Cβ atoms plus residue identity, a different question asked of a different set of atoms entirely. Both would be the natural next step, with a full-atom model instead of this one.